# Daily Returns Artifact — Construction Spec (shared infrastructure)

**Purpose.** This notebook is the **source-of-truth spec** for
`daily_panel_build.ipynb`, which builds the shared daily panel consumed by every
time-series factor build (`beta`, `mom`, `nlb`, `nls`, `resvol`). It is
infrastructure, not a factor: it has no descriptor, no standardisation, and no
USE4 appendix entry of its own — it implements the data conventions the factor
specs share (excess returns, ESTU cap-weighted benchmark).

**Audience.** The assistant building or modifying `daily_panel_build.ipynb`.

**Inputs:** `SHARADAR_SEP.csv`,
`data/out/estu_monthly.parquet` (from `estu_build.ipynb`),
Ken French Data Library (daily RF).

**Deliverables (all `data/out`):** `daily_returns.parquet`,
`crsp_mkt.parquet`, `estu_mkt.parquet`, `ticker_permaticker.parquet`.

**Companion specs:** `estu/estu_spec.ipynb` (upstream),
all five time-series factor guides (downstream).


*(When you productionise, this module is the first thing to move into a real package.)*

## §1. What USE4 requires of this layer — verbatim anchors

### 1a. Excess returns

> *"Excess returns (`r_t − r_ft`) are used"* throughout the USE4 time-series
> estimators (Beta, Momentum, Residual Volatility).

### 1b. Estimation-universe benchmark

> Factor regressions use the **cap-weighted estimation-universe return** —
> the ESTU is the universe over which model parameters are estimated.

---

**That is all the PDF implies for this layer.** Everything else (RF source,
data-quality dedup rules, the FAST/SEQUENTIAL memory strategy) is
❓ **NOT IN PDF** and flagged below.

## §2. End-to-end pipeline

```
┌────────────────────────────────────────────────────────────────────┐
│  UPSTREAM:  estu_build.ipynb  →  estu_monthly.parquet              │
├────────────────────────────────────────────────────────────────────┤
│  STAGE 0:  Setup, parameters                                       │
│  STAGE 1:  Fetch daily RF (Ken French Data Library)                │
│  STAGE 2:  Load prices → excess log returns (FAST/SEQ gate)        │
│  STAGE 3:  FAST ≡ SEQUENTIAL equivalence check                     │
│  STAGE 4:  Load ESTU → cap-weighted excess benchmark               │
│  STAGE 5:  Write artifacts + freshness check                       │
└────────────────────────────────────────────────────────────────────┘
```

**Rebuild order:** run after `estu_build.ipynb` and before any time-series
factor build (one-sweep: `your end-to-end runner`).

**PIT discipline:** returns are computed from same-day and prior-day data only;
`mcap_lag` is the prior trading day's market cap (used as benchmark weight so the
weight is known before the return accrues).

## §3. Artifact schemas — what this notebook delivers

**Compression:** zstd, statistics=True. All files in `data/out`.

### `daily_returns.parquet` — the primary artifact

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `date` | Date | Trading date |
| `ret` | Float64 | **Excess log return**: `ln(closeadj_t / closeadj_{t-1}) − rf_t` |
| `mcap_lag` | Float64 | Prior trading day's market cap (benchmark weight) |
| `mkt_ret` | Float64 | **ESTU cap-weighted excess benchmark** return on `date` |

### Supporting artifacts

| File | Contents |
|---|---|
| `crsp_mkt.parquet` | `date, mkt_ret` — broad CRSP-proxy benchmark (diagnostics only) |
| `estu_mkt.parquet` | `date, estu_mkt_ret` — the ESTU benchmark series on its own |
| `ticker_permaticker.parquet` | `ticker, permaticker` — symbol map for spot-check lookups |

**Coverage rule:** every (permaticker, date) with a valid prior close is present —
the artifact covers the full coverage universe, not just ESTU members.

## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
CALENDAR_START = date(1999, 1, 1)   # matches all factor builds
```

## §5. STAGE 1 — Daily risk-free rate

❓ **NOT IN PDF — RF source.** USE4 does not specify the exact RF series.

💡 **DEFAULT:** Ken French Data Library, 3-Factor daily CSV, `RF` column
(percent → decimal). Over any 252-day window the RF is approximately constant, so
slope coefficients are insensitive to the choice; it is included for fidelity.

🧪 **VALIDATE:** series spans 1926 → present; mean ≈ 3% annualised; daily values
in [0%, 0.07%].

## §6. STAGE 2 — Prices → excess log returns

❓ **NOT IN PDF — data-quality rules.**

💡 **DEFAULT:**
- Dual-class dedup: keep the highest-mcap row per `(permaticker, date)`
- `ret = ln(closeadj_t / closeadj_{t−1}) − rf_t`; rows with null returns dropped
- `mcap_lag = marketcap.shift(1)` per permaticker

❓ **NOT IN PDF — memory strategy.** The full panel is ~39M rows.

💡 **DEFAULT:** commit-aware FAST/SEQUENTIAL gate: FAST (one in-memory
collect+sort) only when free physical RAM ≥ 10× and **commit headroom ≥ 20×** the
on-disk size — other live notebook kernels can exhaust commit even when physical
RAM looks free, and polars aborts the process when an allocation fails.
Otherwise SEQUENTIAL: year-files streamed with a carry row per permaticker so
cross-file returns are exact.

🧪 **VALIDATE:** ~39M rows, ~17,400 tickers, ~6,900 trading days (1998-12 →).

## §7. STAGE 3 — FAST ≡ SEQUENTIAL equivalence

Both load paths must produce identical returns. The check runs both paths on the
first two year-files and asserts:

🧪 **VALIDATE:**
- no SEQ-only rows (FAST-only rows are the first date per ticker — expected)
- max |ret difference| < 1e-12, max |mcap_lag difference| < 1e-12

## §8. STAGE 4 — ESTU cap-weighted excess benchmark

✅ **PDF SPEC:** the benchmark is the cap-weighted return of the estimation
universe (see §1b).

💡 **DEFAULT:** for each trading day, weight member excess returns by `mcap_lag`,
membership taken from the **owning signal date** (most recent month-end ≤ date).
The result replaces the broad CRSP-proxy `mkt_ret` in `daily`; the CRSP proxy is
kept as `crsp_mkt.parquet` for diagnostics.

🧪 **VALIDATE:**
- correlation vs broad CRSP proxy > 0.90
- full-sample annualised vol ≈ 15–20%; GFC and COVID windows > 40%
- dates missing the benchmark after the first signal date < 30

## §9. STAGE 5 — Write artifacts

Write the four artifacts (§3) with zstd compression and read-back shape asserts.

🧪 **VALIDATE — freshness contract:** `daily["date"].max()` ≥ the last ESTU
signal date; otherwise prices must be refreshed before rebuilding. Downstream
loaders (`factor_lib.load_daily_artifacts`) fail loudly if the artifact is
missing.

## §10. Master list of ❓ NOT-IN-PDF decisions

| # | Decision | Default chosen | Revisit when |
|---|---|---|---|
| 1 | RF source | Ken French daily RF | validating against USE4 published numbers |
| 2 | Dual-class dedup | keep highest-mcap row per (permaticker, date) | share-class-level modelling is needed |
| 3 | Benchmark weights | `mcap_lag` (prior day, total mcap — Sharadar has no float) | float data is procured |
| 4 | Memory strategy | commit-aware FAST/SEQ gate (≥10× RAM and ≥20× commit) | hardware changes |
| 5 | Membership mapping | owning signal date = most recent month-end ≤ date | intramonth membership changes matter |

## §11. Final summary — what "done" looks like

1. ✅ Four artifacts in `data/out` with the §3 schemas
2. ✅ FAST ≡ SEQUENTIAL equivalence < 1e-12
3. ✅ Benchmark: CRSP correlation > 0.90, crisis-vol sanity, < 30 missing dates
4. ✅ Freshness: artifact covers the full ESTU signal calendar
5. ✅ All five time-series factor builds load via
   `factor_lib.load_daily_artifacts` without re-deriving anything

Once this passes, the time-series factor builds (`beta`, `mom`, `nlb`, `nls`,
`resvol`) can run in any order (subject to `beta → nlb` and `size → nls`).